# 5.4 자동화 소프트웨어 엔지니어링을 위한 RL


자동화 SWE (Automated Software Engineering):
- **정의**: GitHub 이슈를 받아 자동으로 코드 수정과 PR을 생성하는 에이전트
- **특징**: 다단계 파이프라인 (이슈분석 → 계획 → 구현 → 테스트 → PR)
- **성능**: 실제 프로젝트에서 25-35% 이슈 자동 해결

**전체 파이프라인**:
```
1) Issue Analysis: GitHub 이슈 텍스트 분석
2) Planning: 수정 전략 수립
3) Implementation: 코드 수정
4) Testing: 테스트 실행
5) PR Creation: Pull Request 생성
```

**멀티스테이지 MDP**:
- 각 단계가 자체 상태와 행동을 가짐
- 단계 간 보상 누적
- 실패 시 이전 단계로 롤백 가능

## 멀티스테이지 MDP

### SWE 파이프라인의 MDP 구조

```
메인 에피소드:
  s_0: 이슈 텍스트
    ↓ (이슈분석 에이전트)
  s_1: 분석 결과 + 수정 계획
    ↓ (코드 에이전트)
  s_2: 수정된 코드
    ↓ (테스트 에이전트)
  s_3: 테스트 결과
    ↓ (PR 에이전트)
  s_4: PR 메타데이터

보상:
  r_1 = 이슈분석 정확도 (0~1)
  r_2 = 코드 수정 품질 (0~1)
  r_3 = 테스트 통과율 (0~1)
  r_4 = PR 승인 가능성 (0~1)

최종 보상 = r_1 + r_2 + r_3 + r_4
```

### 계획(Planning) + 도구 사용(Tool Using)의 통합

```
Planning (상위 에이전트):
  이슈를 읽고 전체 해결 전략 수립한다
  예: "파일 A의 함수 X를 수정하고 테스트 Y를 추가한다"

Tool Using (하위 에이전트):
  구체적 도구로 계획 실행한다
  예: 파일 읽기 → 코드 수정 → 파일 쓰기 → 테스트 실행

정책:
  정책 = Planning(이슈) → Tool Using(계획)
  → 큰 전략에서 작은 구현으로
```

In [1]:
from utils_openai import *
import numpy as np
import re

heading("준비: 자동화 SWE RL")
print("✓ 이슈 분석 에이전트")
print("✓ 계획 기반 코드 수정")
print("✓ 테스트 자동 실행")
print("✓ PR 생성 자동화")
print("✓ 멀티스테이지 보상 누적")

def safe_llm_reward(text, criteria="정확성, 완전성, 유용성", max_score=10.0):
    """llm_reward의 안전한 래퍼이다. 숫자 파싱을 보장한다."""
    score_str = ask(
        f"평가 기준: {criteria}\n응답: {text}\n\n0~{max_score} 사이 숫자 하나만 반환하라. 설명 없이 숫자만 출력하라. 예: 7.5",
        temperature=0.0, max_tokens=5
    )
    numbers = re.findall(r'[\d.]+', score_str)
    return min(max(float(numbers[0]), 0.0), max_score) if numbers else max_score * 0.5

def strip_code_blocks(code_text):
    """LLM 응답에서 마크다운 코드 블록을 제거한다."""
    if "```" in code_text:
        code_text = code_text.replace("```python", "").replace("```", "").strip()
    return code_text


────────────────────────────────────────
  준비: 자동화 SWE RL
────────────────────────────────────────
✓ 이슈 분석 에이전트
✓ 계획 기반 코드 수정
✓ 테스트 자동 실행
✓ PR 생성 자동화
✓ 멀티스테이지 보상 누적


## 실습 1: 이슈 분석 에이전트

GitHub 이슈 텍스트를 분석하고 수정 계획을 생성한다.

In [2]:
heading("실습 1: 이슈 분석")

issue = """Title: Bug: ValueError when sorting empty list
Description: When calling sort_numbers([]), the function throws ValueError instead of returning [].
Expected: sort_numbers([]) should return []
Actual: ValueError: list index out of range
Files: utils.py, line 45"""

step_print(1, "이슈", issue)

step_print(2, "이슈 분석", "LLM으로 이슈 해석한다")

analysis_prompt = f"""다음 GitHub 이슈를 분석하고 수정 계획을 수립하라:
{issue}

다음을 제시하라:
1. 문제의 근본 원인
2. 영향받는 코드 위치
3. 수정 전략 (3단계 이하)
4. 테스트할 엣지 케이스"""

analysis = ask(
    analysis_prompt,
    system="소프트웨어 엔지니어이다. 이슈를 분석하고 해결 계획을 제시한다.",
    temperature=0.3
)

print(f"\n  분석 결과:\n{analysis}")

step_print(3, "품질 평가", "분석의 완전성 평가한다")
analysis_quality = safe_llm_reward(analysis, criteria="이슈 분석의 정확성, 명확성, 실행 가능성", max_score=10.0)
print(f"  분석 품질: {analysis_quality:.1f}/10")


────────────────────────────────────────
  실습 1: 이슈 분석
────────────────────────────────────────
  [Step 1] 이슈
    Title: Bug: ValueError when sorting empty list
    Description: When calling sort_numbers([]), the function throws ValueError instead of returning [].
    Expected: sort_numbers([]) should return []
    Actual: ValueError: list index out of range
    Files: utils.py, line 45
  [Step 2] 이슈 분석
    LLM으로 이슈 해석한다

  분석 결과:
이 GitHub 이슈를 분석하고 수정 계획을 수립하겠습니다.

### 1. 문제의 근본 원인
문제의 근본 원인은 `sort_numbers` 함수가 빈 리스트를 처리할 때, 내부적으로 리스트의 인덱스를 접근하려고 시도하면서 발생하는 `ValueError`입니다. 빈 리스트에 대해 인덱스를 참조하려고 하거나, 정렬 알고리즘이 빈 리스트를 처리하는 로직이 누락되어 있을 가능성이 높습니다.

### 2. 영향받는 코드 위치
문제가 발생하는 코드는 `utils.py` 파일의 45번째 줄에 위치한 `sort_numbers` 함수입니다. 이 함수가 빈 리스트를 처리하는 방법이 잘못 구현되어 있습니다.

### 3. 수정 전략 (3단계 이하)
1. **빈 리스트 체크 추가**: `sort_numbers` 함수의 시작 부분에 빈 리스트인지 확인하는 조건문을 추가합니다. 빈 리스트일 경우, 즉시 빈 리스트를 반환하도록 합니다.
   
   ```python
   def sort_numbers(numbers):
       if not numbers:  # 빈 리스트 체크
           return []
  

## 실습 2: 계획 기반 다단계 코드 수정

이슈 분석 결과를 바탕으로 코드를 수정한다.

In [3]:
heading("실습 2: 계획 기반 코드 수정")

original_code = """def sort_numbers(numbers):
    # 버그: 빈 리스트 처리 없음
    sorted_nums = sorted(numbers)
    return sorted_nums[0]  # IndexError!
"""

step_print(1, "원본 코드", original_code)

step_print(2, "계획 수립", "이슈 분석 기반 단계별 계획한다")

plan = """Step 1: 빈 리스트 확인
  if not numbers:
      return []

Step 2: 정렬 및 반환
  return sorted(numbers)

Step 3: 엣지 케이스 테스트
  - 빈 리스트
  - 단일 원소
  - 중복 원소"""

print(f"  {plan}")

step_print(3, "코드 수정 생성", "계획을 코드로 구현한다")

fix_prompt = f"""다음 버그 있는 코드를 수정하라:
{original_code}

수정 계획:
{plan}

설명 없이 수정된 함수만 반환하라."""

fixed_code = ask(fix_prompt, system="파이썬 개발자이다. 정확한 함수만 작성한다.", temperature=0.3)
fixed_code = strip_code_blocks(fixed_code)

print(f"  수정된 코드:\n{fixed_code}")

step_print(4, "코드 품질 평가", "수정된 코드 검토한다")
fix_quality = safe_llm_reward(fixed_code, criteria="코드 정확성, 버그 처리, 가독성", max_score=10.0)
print(f"  수정 품질: {fix_quality:.1f}/10")


────────────────────────────────────────
  실습 2: 계획 기반 코드 수정
────────────────────────────────────────
  [Step 1] 원본 코드
    def sort_numbers(numbers):
        # 버그: 빈 리스트 처리 없음
        sorted_nums = sorted(numbers)
        return sorted_nums[0]  # IndexError!
    
  [Step 2] 계획 수립
    이슈 분석 기반 단계별 계획한다
  Step 1: 빈 리스트 확인
  if not numbers:
      return []

Step 2: 정렬 및 반환
  return sorted(numbers)

Step 3: 엣지 케이스 테스트
  - 빈 리스트
  - 단일 원소
  - 중복 원소
  [Step 3] 코드 수정 생성
    계획을 코드로 구현한다
  수정된 코드:
def sort_numbers(numbers):
    if not numbers:
        return []
    return sorted(numbers)
  [Step 4] 코드 품질 평가
    수정된 코드 검토한다
  수정 품질: 3.2/10


## 실습 3: 자동화 SWE의 보상 체계

각 단계별 보상을 정의하고 누적한다.

In [4]:
heading("실습 3: 멀티스테이지 보상")

step_print(1, "보상 체계", "SWE 파이프라인의 4단계 보상한다")

rewards = {
    "분석 (Analysis)": {"기준": "이슈 분석의 정확성과 명확성", "점수": 8.5, "가중치": 0.2},
    "구현 (Implementation)": {"기준": "코드 수정의 정확성과 품질", "점수": 9.2, "가중치": 0.3},
    "테스트 (Testing)": {"기준": "테스트 통과율과 엣지 케이스 처리", "점수": 9.0, "가중치": 0.3},
    "PR (Pull Request)": {"기준": "PR 설명, 제목, 메타데이터 품질", "점수": 8.0, "가중치": 0.2}
}

step_print(2, "단계별 점수", "각 단계별 평가 결과한다")

total_weighted_score = 0
for stage, data in rewards.items():
    weighted = data['점수'] * data['가중치']
    total_weighted_score += weighted
    print(f"  {stage}")
    print(f"    점수: {data['점수']:.1f}/10")
    print(f"    가중치: {data['가중치']}")
    print(f"    가중 점수: {weighted:.2f}")

step_print(3, "최종 점수", "전체 보상 계산한다")
print(f"  최종 점수: {total_weighted_score:.2f}/10")
print(f"  평가: {'우수' if total_weighted_score >= 8.5 else '양호' if total_weighted_score >= 7 else '개선필요'}")
print(f"  → 이슈 자동 해결 성공 가능성: {total_weighted_score/10:.0%}")


────────────────────────────────────────
  실습 3: 멀티스테이지 보상
────────────────────────────────────────
  [Step 1] 보상 체계
    SWE 파이프라인의 4단계 보상한다
  [Step 2] 단계별 점수
    각 단계별 평가 결과한다
  분석 (Analysis)
    점수: 8.5/10
    가중치: 0.2
    가중 점수: 1.70
  구현 (Implementation)
    점수: 9.2/10
    가중치: 0.3
    가중 점수: 2.76
  테스트 (Testing)
    점수: 9.0/10
    가중치: 0.3
    가중 점수: 2.70
  PR (Pull Request)
    점수: 8.0/10
    가중치: 0.2
    가중 점수: 1.60
  [Step 3] 최종 점수
    전체 보상 계산한다
  최종 점수: 8.76/10
  평가: 우수
  → 이슈 자동 해결 성공 가능성: 88%


## 실습 4: 전체 SWE 파이프라인 시뮬레이션

이슈 분석부터 PR 생성까지 전체 프로세스를 시뮬레이션한다.

In [5]:
heading("실습 4: 전체 SWE 파이프라인")

pipeline_issue = """Title: Feature: Add password validation
Description: Add password strength validation to User signup form.
Criteria:
- At least 8 characters
- At least one uppercase
- At least one digit
- At least one special character"""

step_print(1, "입력", "이슈 텍스트")
print(f"  {pipeline_issue}")

step_print(2, "1단계: 분석", "이슈 파악한다")
analysis_result = ask(
    f"다음 이슈를 분석하고 구현 계획을 제시하라:\n{pipeline_issue}",
    system="소프트웨어 엔지니어이다.",
    temperature=0.3
)
print(f"  분석 완료: {len(analysis_result)} 글자")
analysis_score = safe_llm_reward(analysis_result, criteria="이슈 분석의 정확성", max_score=10.0)
print(f"  분석 점수: {analysis_score:.1f}/10")

step_print(3, "2단계: 구현", "코드 작성한다")
impl_result = ask(
    f"다음 요구사항으로 password validation 함수를 작성하라. 함수명: validate_password, 입력: str, 출력: bool\n{pipeline_issue}\n설명 없이 함수만 반환하라.",
    system="파이썬 개발자이다.",
    temperature=0.3
)
impl_result = strip_code_blocks(impl_result)
print(f"  구현 완료: {len(impl_result)} 글자")
impl_score = safe_llm_reward(impl_result, criteria="코드 정확성과 완전성", max_score=10.0)
print(f"  구현 점수: {impl_score:.1f}/10")

step_print(4, "3단계: 테스트", "엣지 케이스 검증한다")
namespace = {}
exec(impl_result, namespace)
validate_pw = namespace.get('validate_password')

test_cases = [
    ("Pass123!", True),
    ("Pass123", False),
    ("password", False),
    ("P1!", False)
]

test_passed = 0
if validate_pw:
    for pw, expected in test_cases:
        result = validate_pw(pw)
        status = "✓" if result == expected else "✗"
        if result == expected:
            test_passed += 1
        print(f"  validate_password('{pw}') = {result} (기대: {expected}) {status}")

test_score = (test_passed / len(test_cases)) * 10
print(f"  테스트 점수: {test_score:.1f}/10")

step_print(5, "4단계: PR 생성", "Pull Request 메타데이터")
pr_data = {
    "title": "feat: Add password strength validation",
    "description": "Implements password strength validation with multiple criteria checks.",
    "labels": ["feature", "security"]
}
pr_score = 8.0
print(f"  PR 제목: {pr_data['title']}")
print(f"  PR 설명: {pr_data['description']}")
print(f"  라벨: {', '.join(pr_data['labels'])}")
print(f"  PR 점수: {pr_score:.1f}/10")

step_print(6, "최종 결과", "파이프라인 성공률")
final_score = analysis_score * 0.2 + impl_score * 0.3 + test_score * 0.3 + pr_score * 0.2
print(f"  분석: {analysis_score:.1f} (20%)")
print(f"  구현: {impl_score:.1f} (30%)")
print(f"  테스트: {test_score:.1f} (30%)")
print(f"  PR: {pr_score:.1f} (20%)")
print(f"\n  최종 점수: {final_score:.2f}/10")
print(f"  자동 해결 확률: {final_score/10:.0%}")
print(f"  상태: {'✓ 성공 가능' if final_score >= 8 else '△ 검토 필요' if final_score >= 6 else '✗ 재시도'}")


────────────────────────────────────────
  실습 4: 전체 SWE 파이프라인
────────────────────────────────────────
  [Step 1] 입력
    이슈 텍스트
  Title: Feature: Add password validation
Description: Add password strength validation to User signup form.
Criteria:
- At least 8 characters
- At least one uppercase
- At least one digit
- At least one special character
  [Step 2] 1단계: 분석
    이슈 파악한다
  분석 완료: 2972 글자
  분석 점수: 9.0/10
  [Step 3] 2단계: 구현
    코드 작성한다
  구현 완료: 282 글자
  구현 점수: 9.0/10
  [Step 4] 3단계: 테스트
    엣지 케이스 검증한다
  validate_password('Pass123!') = True (기대: True) ✓
  validate_password('Pass123') = False (기대: False) ✓
  validate_password('password') = False (기대: False) ✓
  validate_password('P1!') = False (기대: False) ✓
  테스트 점수: 10.0/10
  [Step 5] 4단계: PR 생성
    Pull Request 메타데이터
  PR 제목: feat: Add password strength validation
  PR 설명: Implements password strength validation with multiple criteria checks.
  라벨: feature, security
  PR 점수: 8.0/10
  [Step 6] 최종 결과
    파이프라인 성공률
  분석: 9.0 (20%)


## 요약: 자동화 소프트웨어 엔지니어링 RL

### 전체 파이프라인

```
GitHub Issue
    ↓
1. Issue Analysis (20% 가중치)
   - 문제 파악한다
   - 수정 계획 수립한다
    ↓
2. Implementation (30% 가중치)
   - 코드 작성한다
   - 품질 검증한다
    ↓
3. Testing (30% 가중치)
   - 자동 테스트 실행한다
   - 엣지 케이스 확인한다
    ↓
4. PR Generation (20% 가중치)
   - Pull Request 메타데이터 생성한다
   - 설명과 레이블 작성한다
    ↓
최종 점수 = Σ (단계 점수 × 가중치)
```

### 멀티스테이지 MDP의 특징

1. **계층적 정책**
   - 상위: 전체 해결 전략 (Planning)
   - 하위: 구체적 행동 (Implementation)

2. **단계 간 정보 흐름**
   - 각 단계의 출력 = 다음 단계의 입력
   - 오류 전파 관리 필요

3. **누적 보상**
   - 각 단계에서 부분 점수 획득
   - 최종 점수 = 모든 단계의 가중합

### 실무 성능

- **자동 해결 가능 비율**: 25-35% (실제 오픈소스 프로젝트)
- **부분 해결 비율**: 40-50% (계속 개선 필요)
- **실패율**: 15-25% (인간 검토 필요)

### 성능 향상 전략

1. **더 나은 모델**: GPT-4 사용한다 (30-40% 개선)
2. **외부 도구**: 정적 분석, 린트 통합한다
3. **메모리 활용**: 과거 유사 이슈 사례 학습한다
4. **피드백 루프**: 실패한 경우 이유 분석 및 학습한다